In [1]:
from datasets import load_dataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

print("Loading dataset...")
dataset = load_dataset("google/civil_comments")

# Convert splits to dataframes
train_df = pd.DataFrame(dataset['train'])
test_df = pd.DataFrame(dataset['test'])

# Define toxicity binary label (threshold >= 0.5)
train_df['label'] = (train_df['toxicity'] >= 0.5).astype(int)
test_df['label'] = (test_df['toxicity'] >= 0.5).astype(int)

# FIX: Changed 'comment_text' to 'text' to match the exact Hugging Face column layout
train_df['text'] = train_df['text'].fillna("")
test_df['text'] = test_df['text'].fillna("")

print("Performing foolproof stratified sampling...")
# Standardizing the size to 150,000 train samples while keeping perfectly balanced class allocations
_, train_sampled = train_test_split(
    train_df, 
    test_size=150000, 
    stratify=train_df['label'], 
    random_state=42
)

# Standardizing the size to 30,000 test samples
_, test_sampled = train_test_split(
    test_df, 
    test_size=30000, 
    stratify=test_df['label'], 
    random_state=42
)

# Reset indexes cleanly to eliminate index alignment bugs
train_sampled = train_sampled.reset_index(drop=True)
test_sampled = test_sampled.reset_index(drop=True)

# Extract inputs and targets safely
X_train, y_train = train_sampled['text'], train_sampled['label']
X_test, y_test = test_sampled['text'], test_sampled['label']

# Save splits to disk with the standardized names 'text' and 'label'
train_sampled[['text', 'label']].to_csv('train_split.csv', index=False)
test_sampled[['text', 'label']].to_csv('test_split.csv', index=False)

print("Data sampled and saved successfully with no errors!")
print(f"Final Train Value Counts:\n{y_train.value_counts()}")

c:\Users\BLACKBOX\.anaconda-desktop\micromamba\envs\cuda\envs\condaenv1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading dataset...


c:\Users\BLACKBOX\.anaconda-desktop\micromamba\envs\cuda\envs\condaenv1\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\BLACKBOX\.cache\huggingface\hub\datasets--google--civil_comments. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|██████████| 97320/97320 [00:00<00:00, 1624

Performing foolproof stratified sampling...
Data sampled and saved successfully with no errors!
Final Train Value Counts:
label
0    138005
1     11995
Name: count, dtype: int64


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

print("Vectorizing text data...")
tfidf = TfidfVectorizer(max_features=15000, ngram_range=(1, 2), stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Save the vectorizer to disk
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')

Vectorizing text data...


['tfidf_vectorizer.pkl']

In [6]:
import time
import os
import psutil
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score


def track_inference_efficiency(model, X_data):
    process = psutil.Process(os.getpid())
    start_mem = process.memory_info().rss / (1024 * 1024) # MB
    
    start_time = time.perf_counter()
    predictions = model.predict(X_data)
    end_time = time.perf_counter()
    
    end_mem = process.memory_info().rss / (1024 * 1024) # MB
    
    total_latency_ms = (end_time - start_time) * 1000
    latency_per_sample_ms = total_latency_ms / X_data.shape[0]
    
    return predictions, latency_per_sample_ms, (end_mem - start_mem)

# Train Logistic Regression
print("Training Logistic Regression...")
lr_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_model.fit(X_train_tfidf, y_train)
joblib.dump(lr_model, 'logistic_regression_model.pkl')

# Train Linear SVC
print("Training Linear SVC...")
svc_model = LinearSVC(class_weight='balanced', random_state=42)
svc_model.fit(X_train_tfidf, y_train)
joblib.dump(svc_model, 'linear_svc_model.pkl')

# Benchmark Models
for name, model in [("Logistic Regression", lr_model), ("Linear SVC", svc_model)]:
    preds, latency, mem = track_inference_efficiency(model, X_test_tfidf)
    f1 = f1_score(y_test, preds)
    print(f"\n[{name}] Results:")
    print(f"Base F1 Score: {f1:.4f}")
    print(f"Latency per Document: {latency:.4f} ms")
    print(f"Memory Overhead: {mem:.2f} MB")

Training Logistic Regression...
Training Linear SVC...

[Logistic Regression] Results:
Base F1 Score: 0.5516
Latency per Document: 0.0001 ms
Memory Overhead: 0.00 MB

[Linear SVC] Results:
Base F1 Score: 0.5220
Latency per Document: 0.0000 ms
Memory Overhead: 0.00 MB
